# BAMO — Persian Natural TTS (Colab GPU)
مدل: `facebook/mms-tts-fas` (فارسی)
خروجی: فایل‌های wav جمله‌به‌جمله + narration کامل

In [ ]:
!pip -q install -U transformers accelerate scipy soundfile torch torchvision torchaudio
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch, numpy as np, scipy.io.wavfile as wavfile
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = VitsModel.from_pretrained('facebook/mms-tts-fas').to(device)
tokenizer = AutoTokenizer.from_pretrained('facebook/mms-tts-fas')
sr = model.config.sampling_rate
print('ready', sr, device)

In [ ]:
# متن شورت ریاضی ۰٫۹۹۹… = ۱ — نگارش طبیعی برای TTS
BEATS = [
    'صبر کن. این یکی عجیبه.',
    'صفر ممیز نه نه نه، تا بی نهایت.',
    'خیلی ها فکر می کنند از یک کمتر است.',
    'اما برابر یک است.',
    'فرض کن ایکس بشود صفر ممیز نه تکراری.',
    'ده برابرش می شود نه ممیز نه تکراری. بعد کم کن.',
    'نه ایکس می شود نه. پس ایکس می شود یک.',
    'ریاضی گاهی خلاف حس است، اما درست است.',
    'بامو. یوزر نیم ما دوازده بامو.',
]

out_dir = Path('bamo_tts_out')
out_dir.mkdir(exist_ok=True)

def synth(text: str):
    inputs = tokenizer(text, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        wave = model(**inputs).waveform
    audio = wave.squeeze().cpu().numpy()
    # normalize lightly
    peak = np.max(np.abs(audio)) + 1e-8
    audio = (audio / peak * 0.95).astype(np.float32)
    return audio

parts = []
for i, text in enumerate(BEATS):
    audio = synth(text)
    path = out_dir / f'p{i:02d}.wav'
    wavfile.write(str(path), rate=sr, data=audio)
    parts.append(audio)
    print(i, f'{len(audio)/sr:.2f}s', text)
    # micro silence between sentences
    gap = np.zeros(int(sr * 0.12), dtype=np.float32)
    parts.append(gap)

full = np.concatenate(parts)
wavfile.write(str(out_dir / 'narration.wav'), rate=sr, data=full)
print('saved', out_dir / 'narration.wav', f'{len(full)/sr:.2f}s')

In [ ]:
from IPython.display import Audio, display
display(Audio(str(out_dir / 'narration.wav')))
# دانلود فایل ها از پنل سمت چپ Colab → bamo_tts_out/